# Data modeling and query optimization

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-09-modeling-and-optimization.ipynb)

Two decisions shape almost every analytics query: how the data is **modeled**, and how the engine **executes** it.
Good modeling makes the common question a simple join. Good optimization makes the join cheap. This chapter covers
the grain of a table, the star schema, slowly changing dimensions, reading a query plan, and filters the engine can
use. It ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW segment_history AS
SELECT * FROM VALUES
  (1, 'standard', TIMESTAMP '2026-01-01 00:00:00', TIMESTAMP '2026-06-01 00:00:00'),
  (1, 'premium',  TIMESTAMP '2026-06-01 00:00:00', NULL),
  (2, 'standard', TIMESTAMP '2026-01-01 00:00:00', NULL)
AS segment_history(customer_id, segment, effective_from, effective_to)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW txns AS
SELECT * FROM VALUES
  (1, 1,  TIMESTAMP '2026-01-15 12:00:00', 20.0),
  (2, 1,  TIMESTAMP '2026-07-01 12:00:00', 30.0),
  (3, 2,  TIMESTAMP '2026-03-01 12:00:00', 40.0),
  (4, 99, TIMESTAMP '2026-03-01 12:00:00', 50.0),
  (5, 1,  TIMESTAMP '2026-06-01 00:00:00', 10.0)
AS txns(txn_id, customer_id, event_ts, amount)
""")

## 1. Start with the grain

Before you model a table, write its **grain** as one sentence: "one row per order", "one row per customer per day",
"one row per change of segment". Every key and every join must agree with that sentence. Most modeling mistakes are
tables whose grain was never written down.

## 2. The star schema

A **star schema** has one **fact** table at a declared grain, with **dimension** tables around it that describe its
keys. `orders` is a fact table at the grain "one row per order". `customers` is a dimension keyed by `customer_id`.

Joining a fact table to a dimension must not change the fact's row count. That holds only when the dimension key is
unique. Check it:

In [ ]:
spark.sql("""
SELECT COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_keys
FROM customers
""").show()

spark.sql("""
SELECT COUNT(*) AS fact_rows
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
""").show()

## 3. Slowly changing dimensions (Type 2)

Customer attributes change over time, and history must be kept. A Type 2 dimension stores one row per version, with
`effective_from` and `effective_to`. The interval is **half-open**: `[effective_from, effective_to)`. A NULL end means
the version is still current.

To find the version that applied to each transaction, join on a **range**. Half-open intervals make the boundary
unambiguous: a transaction exactly at `effective_to` belongs to the next version. Transaction 5 below falls on that
boundary and gets `premium`.

In [ ]:
spark.sql("""
SELECT t.txn_id, t.customer_id, s.segment
FROM txns t
LEFT JOIN segment_history s
  ON s.customer_id = t.customer_id
 AND t.event_ts >= s.effective_from
 AND (s.effective_to IS NULL OR t.event_ts < s.effective_to)
ORDER BY t.txn_id
""").show()

Transaction 4 has no matching customer, so it keeps a NULL segment. A left join keeps it.

Before you trust a Type 2 table, check that no two versions for the same key overlap. An overlap would make the join
above return two rows for one transaction.

In [ ]:
spark.sql("""
SELECT COUNT(*) AS overlapping_pairs
FROM segment_history a
JOIN segment_history b
  ON a.customer_id = b.customer_id
 AND a.effective_from < b.effective_from
 AND b.effective_from < COALESCE(a.effective_to, TIMESTAMP '9999-12-31 00:00:00')
""").show()

## 4. Reading a query plan

`EXPLAIN FORMATTED` shows how Spark will run a query. Look at the join strategy and at the `Exchange` nodes, which
are shuffles across the network. A `BroadcastHashJoin` sends the small table to every executor and avoids shuffling
the large one. The `BROADCAST` hint asks Spark to do that. The result is the same either way. The plan is what
changes.

In [ ]:
QUERY = """
SELECT /*+ BROADCAST(c) */ c.region, SUM(o.amount) AS completed_total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
GROUP BY c.region
ORDER BY c.region
"""
spark.sql(QUERY).show()

plan = spark.sql("EXPLAIN FORMATTED " + QUERY).first()[0]
assert "BroadcastHashJoin" in plan

## 5. Filters the engine can use

A filter is **sargable** when the engine can use it to skip data, for example a comparison of a bare column with a
constant. Wrapping the column in a function usually forces a scan of every row. On a large partitioned table, only the
first form below can skip whole partitions. Both return the same five February orders.

In [ ]:
spark.sql("""
SELECT COUNT(*) AS february_orders
FROM orders
WHERE order_date >= DATE '2023-02-01' AND order_date < DATE '2023-03-01'
""").show()

spark.sql("""
SELECT COUNT(*) AS february_orders
FROM orders
WHERE date_format(order_date, 'yyyy-MM') = '2023-02'
""").show()

## 6. Nested values

Spark can keep collections and structs inside one column. `collect_set` gathers the distinct values of a group into
an array, and `named_struct` builds a struct whose field you can read. They suit denormalised reporting tables.

In [ ]:
spark.sql("""
SELECT customer_id, sort_array(collect_set(status)) AS statuses
FROM orders
WHERE customer_id = 2
GROUP BY customer_id
""").show(truncate=False)

spark.sql("""
SELECT named_struct('id', order_id, 'amount', amount).amount AS amount
FROM orders
WHERE order_id = 1
""").show()

## Summary

- Write the grain first, and make every key and join agree with it.
- Dimension keys must be unique. Check it.
- Model changing attributes as half-open Type 2 intervals, and test that they do not overlap.
- Read the plan before you tune. It shows which join was chosen and where the shuffles are.

## Practice questions

1. How many completed orders does each region have? Use a join to `customers`, sorted by region. Customers 7 and 8 have no completed orders, so they do not appear.
2. Which segment applied to transaction 2, and which to transaction 3? Show `txn_id` and `segment`, sorted by `txn_id`.
3. Which distinct statuses does customer 5 have, as one array? Use `sort_array`.

## Answers

### Question 1

In [ ]:
q = """
SELECT c.region, SUM(o.amount) AS completed_total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
GROUP BY c.region
ORDER BY c.region
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("North", 1150), ("South", 620), ("West", 1225)]

### Question 2

In [ ]:
q = """
SELECT t.txn_id, s.segment
FROM txns t
LEFT JOIN segment_history s
  ON s.customer_id = t.customer_id
 AND t.event_ts >= s.effective_from
 AND (s.effective_to IS NULL OR t.event_ts < s.effective_to)
WHERE t.txn_id IN (2, 3)
ORDER BY t.txn_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, "premium"), (3, "standard")]

### Question 3

In [ ]:
q = """
SELECT customer_id, sort_array(collect_set(status)) AS statuses
FROM orders
WHERE customer_id = 5
GROUP BY customer_id
"""
spark.sql(q).show(truncate=False)
assert spark.sql(q).collect() == [(5, ["completed"])]